# Layer 2: KNN-Shift — Transformer (Retrained)

Behavioral reclassification test (KNN K=15). Each retrained model has its own embedding space,
so we build a self-contained KNN per model and compare only scalar fractions across models.

- **Infiltration**: fake users' fraction of K nearest neighbors that are target-genre.
- **Displacement**: real users' neighborhood fraction change vs ORIGINAL baseline.

Output: `layer2/result/knn_shift_retrained_summary.csv` + figures.

In [ ]:
import pickle, ast, re
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

BASE = Path('/home/moshtasa/Research/phd-recsys/Book/current/PerturbRec')
ORIGINAL_CSV = BASE / 'Data' / 'ORIGINAL_books.csv'
TDIR = BASE / 'transformer'
EMB_DIR = TDIR / 'result' / 'embeddings'          # retrained embeddings (shared input)

USER_COL, BOOK_COL, GENRE_COL = 'user_id', 'book_id', 'genres'
GENRES = ['Drama']
N_FILTER = {2, 4, 6, 25, 50, 100, 200, 500, 1000, 2000}
PATTERN = re.compile(r'^f_(.+)_(\d+)u_pos(\d+)_neg(NA|0|1|\d+)_.+\.csv$')

print('Config loaded. Embeddings:', EMB_DIR)


In [ ]:
## Shared utilities

def parse_genres(s):
    if pd.isna(s): return []
    s = str(s).strip()
    if not s: return []
    if (s.startswith('[') and s.endswith(']')) or (s.startswith('(') and s.endswith(')')):
        try:
            p = ast.literal_eval(s)
            if isinstance(p, (list, tuple)):
                return [str(x).strip().strip('"').strip("'") for x in p if str(x).strip()]
        except: pass
    for sep in [',', '|', ';', '//', '/']:
        if sep in s:
            return [t.strip().strip('"').strip("'") for t in s.split(sep) if t.strip()]
    return [s.strip().strip('"').strip("'")]

def book_genres_map():
    df = pd.read_csv(ORIGINAL_CSV, usecols=[BOOK_COL, GENRE_COL], low_memory=False)
    df = df.drop_duplicates(subset=[BOOK_COL])
    df[GENRE_COL] = df[GENRE_COL].fillna('').astype(str)
    return {int(r[BOOK_COL]): parse_genres(r[GENRE_COL]) for _, r in df.iterrows()}

def user_dominant_genre_map(bg):
    df = pd.read_csv(ORIGINAL_CSV, usecols=[USER_COL, BOOK_COL], low_memory=False)
    df[USER_COL] = df[USER_COL].astype(int)
    df[BOOK_COL] = df[BOOK_COL].astype(int)
    out = {}
    for uid, g in df.groupby(USER_COL):
        gs = []
        for b in g[BOOK_COL]: gs.extend(bg.get(int(b), []))
        out[int(uid)] = Counter(gs).most_common(1)[0][0] if gs else 'Unknown'
    return out

def load_art(path):
    with open(path, 'rb') as f: return pickle.load(f)

def scan_injections(emb_dir):
    jobs = []
    for fp in sorted(emb_dir.glob('f_*_model.pt')):
        stem = fp.stem.replace('_model', '')
        m = PATTERN.match(stem + '.csv')
        if m and int(m.group(3)) == 5 and int(m.group(2)) in N_FILTER and m.group(1) in GENRES:
            jobs.append((fp, stem, m.group(1), int(m.group(2))))
    return sorted(jobs, key=lambda x: (x[2], x[3]))

bg = book_genres_map()
ulabels = user_dominant_genre_map(bg)
orig_uids_set = set(ulabels.keys())
print(f'Loaded {len(ulabels)} original user labels')


In [ ]:
from sklearn.neighbors import NearestNeighbors

OUT = TDIR / 'layer2' / 'result'
(OUT / 'figure').mkdir(parents=True, exist_ok=True)
K = 15
print('KNN K =', K)


In [ ]:
## Build ORIGINAL baseline (per-user frac) + run all injections

art0 = load_art(EMB_DIR / 'ORIGINAL_model.pt')
pu0 = art0['pu'].astype(np.float32)
uids0 = [int(u) for u in art0['inner_uid_to_raw']]
labels0 = np.array([ulabels.get(u, 'Unknown') for u in uids0])
nbrs0 = NearestNeighbors(n_neighbors=K + 1).fit(pu0)
_, idx0 = nbrs0.kneighbors(pu0)

base_per_user = {}
for tgt in GENRES:
    base_per_user[tgt] = {uids0[i]: float(np.mean(labels0[idx0[i, 1:K+1]] == tgt)) for i in range(len(uids0))}
    print(f'  baseline mean frac_{tgt} = {np.mean(list(base_per_user[tgt].values())):.4f}')

results = []
for fp, name, genre, n in scan_injections(EMB_DIR):
    art = load_art(fp)
    pu = art['pu'].astype(np.float32)
    uids = [int(u) for u in art['inner_uid_to_raw']]
    labels = np.array([ulabels.get(u, genre) for u in uids])
    fake_mask = np.array([u not in orig_uids_set for u in uids])
    real_mask = ~fake_mask

    nbrs = NearestNeighbors(n_neighbors=K + 1).fit(pu)
    _, idx = nbrs.kneighbors(pu)
    real_disp, fake_infil = [], []
    for j in range(len(uids)):
        neigh = idx[j, 1:K+1]
        frac = float(np.mean(labels[neigh] == genre))
        if real_mask[j]:
            real_disp.append(frac - base_per_user[genre].get(uids[j], 0.0))
        else:
            fake_infil.append(frac)
    disp = float(np.mean(real_disp)) if real_disp else np.nan
    infil = float(np.mean(fake_infil)) if fake_infil else np.nan
    results.append({'base': name, 'genre': genre, 'n': n, 'infiltration': infil, 'displacement': disp, 'n_biasing': int(fake_mask.sum()), 'n_real': int(real_mask.sum())})
    print(f'  {name}: infil={infil:.4f} disp={disp:+.4f}')

df = pd.DataFrame(results)
df.to_csv(OUT / 'knn_shift_retrained_summary.csv', index=False)
df.head()


In [ ]:
## Plot infiltration + displacement vs n

colors = {'Drama': 'steelblue'}
df = df.sort_values(['genre', 'n'])
for metric, ylabel, title in [('displacement', 'displacement (real users shift)', 'Displacement'), ('infiltration', 'infiltration (fake users frac)', 'Infiltration')]:
    fig, ax = plt.subplots(figsize=(8, 6))
    for g, s in df.groupby('genre'):
        ax.plot(s['n'], s[metric], 'o-', c=colors.get(g, 'gray'), label=g)
    ax.axhline(0, color='gray', lw=0.8, ls='--')
    ax.set_xscale('log'); ax.set_xlabel('Injected biasing users (n)')
    ax.set_ylabel(ylabel); ax.set_title(f'Layer 2: {title} vs n (K={K})')
    ax.legend(); fig.tight_layout()
    fig.savefig(OUT / 'figure' / f'{metric}_vs_n.png', dpi=150)
    plt.close(fig)
    print(f'  saved -> {metric}_vs_n.png')

print('Layer 2 complete.')
